# Hyperparameter Tuning — CNN-1D URL Classifier
**Google Colab | Grid Search | Dataset 50.000 Balanced**

---
Notebook ini mencari kombinasi hyperparameter terbaik untuk model CNN-1D
berbasis karakter untuk klasifikasi URL menggunakan **Grid Search** (mencoba semua kombinasi).

**Arsitektur CNN-1D (Sequential — satu lapisan konvolusi):**
```
Input [MAX_LEN, int32]
  -> Embedding(vocab=41, dim=32)
  -> Conv1D(num_filters, kernel_size, relu)
  -> MaxPooling1D
  -> Dropout
  -> GlobalMaxPooling1D
  -> Dense(64, relu)
  -> Dense(1, sigmoid)
```

**Hyperparameter yang dicari (sesuai Tabel III.3 laporan):**
- `num_filters`: jumlah filter Conv1D -> [32, 64, 128]
- `kernel_size`: ukuran kernel konvolusi -> [3, 5, 7]
- `dropout_rate`: tingkat dropout -> [0.1, 0.2, 0.5]
- `learning_rate`: laju belajar optimizer Adam -> [0.001, 0.0001]

Total kombinasi: 3 x 3 x 3 x 2 = **54 kombinasi** (semua dicoba).

**Output per eksperimen:**
- `training_history.png` — plot loss & accuracy per epoch
- `metrics_barchart.png` — bar chart Accuracy/Precision/Recall/F1
- `experiment_info.json` — metadata lengkap eksperimen
- `epoch_history.csv` — log metrik per epoch

> Menggunakan **50K** data (sample) untuk mempercepat tuning.
> Training final dengan dataset 100rb/200rb/300rb ada di notebook `cnn1d_training.ipynb`.

## Cell 1 — Install Library

In [ ]:
!pip install -q --upgrade scikit-learn
print('Install selesai.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import re
import time
import random
import itertools
import datetime
import warnings
warnings.filterwarnings('ignore')

# Reproduksibilitas penuh
os.environ['PYTHONHASHSEED'] = '0'
random.seed(42)
np.random.seed(42)

import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    classification_report
)
from sklearn.utils.class_weight import compute_class_weight

tf.random.set_seed(42)

print(f'TensorFlow   : {tf.__version__}')
print(f'GPU tersedia : {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'Seed         : 42 (NumPy, TF, Python random, PYTHONHASHSEED=0)')

## Cell 3 — Mount Google Drive & Konfigurasi

**Format dataset yang diharapkan** — CSV dengan kolom:
```
url, label   (atau)   domain, label
```
> `label`: 0 = URL aman, 1 = URL pornografi

Jika dataset sudah memiliki kolom fitur RF (`domain_length`, dst.),
notebook ini akan menggunakan kolom `url` untuk CNN-1D.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_PATH = '/content/drive/MyDrive/Tugas Akhir/Dataset/dataset_50rb.csv'
SAVE_PATH    = '/content/drive/MyDrive/Tugas Akhir/Tuning/CNN1D/'
URL_COL      = 'url'    # Nama kolom URL atau domain di dataset
LABEL_COL    = 'label'
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)

df = pd.read_csv(DATASET_PATH)
print(f'Dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Distribusi label:\n{df[LABEL_COL].value_counts()}')
df[[URL_COL, LABEL_COL]].head()

## Cell 4 — Fungsi Tokenisasi

> **PENTING**: Semua fungsi ini HARUS IDENTIK dengan kode di `UrlClassifier.kt`.

**CHAR_TO_IDX** — pemetaan karakter ke indeks:
```
0      → PAD (karakter kosong)
1      → UNK (karakter tidak dikenal)
2–27   → a–z
28–37  → 0–9
38     → titik (.)
39     → hubung (-)
40     → garis bawah (_)
```
Total vocab: 41 karakter.

In [ ]:
# Bangun CHAR_TO_IDX — identik dengan Android UrlClassifier.kt
CHAR_TO_IDX = {chr(0): 0}  # PAD = indeks 0
for i, c in enumerate('abcdefghijklmnopqrstuvwxyz'):
    CHAR_TO_IDX[c] = i + 2       # a=2, b=3, ..., z=27
for i, c in enumerate('0123456789'):
    CHAR_TO_IDX[c] = i + 28      # 0=28, 1=29, ..., 9=37
CHAR_TO_IDX['.'] = 38
CHAR_TO_IDX['-'] = 39
CHAR_TO_IDX['_'] = 40

UNK_IDX    = 1
PAD_IDX    = 0
VOCAB_SIZE = 41  # indeks 0–40

# TLD yang dikenal (dipakai untuk ekstrak core domain)
COMMON_TLDS = {
    'com','net','org','info','biz','co','io','me','tv','cc','in','ru','cn',
    'jp','kr','de','uk','fr','it','es','br','au','ca','nl','id','my','th',
    'ph','vn','sg','hk','tw','xyz','top','site','online','club','live',
    'fun','space','tech','store','shop','app','dev','porn','sex','xxx',
    'adult','cam','tube','tk','ml','ga','cf','gq','vip','pw'
}
SECOND_LEVEL_TLDS = {
    'co.id','co.uk','co.jp','co.kr','com.au','com.br','com.cn','com.hk',
    'com.my','com.sg','com.tw','com.vn','net.id','ac.id','go.id','web.id'
}

def normalize_domain(url: str) -> str:
    """Normalisasi URL ke domain — identik dengan Android normalizeDomain()"""
    url = url.lower().strip()
    for prefix in ['https://', 'http://', 'www.']:
        if url.startswith(prefix):
            url = url[len(prefix):]
    url = url.split('/')[0].split('?')[0].split('#')[0].split(':')[0]
    return url

def extract_core_domain(full_domain: str) -> str:
    """Ekstrak nama inti domain — identik dengan Android extractMainDomainName()"""
    parts = full_domain.split('.')
    if len(parts) < 2:
        return full_domain
    if len(parts) >= 3:
        potential_2nd = f'{parts[-2]}.{parts[-1]}'
        if potential_2nd in SECOND_LEVEL_TLDS:
            return parts[-3]
    if parts[-1] in COMMON_TLDS:
        return parts[-2]
    return parts[-2] if len(parts) >= 2 else full_domain

def tokenize(domain: str, max_len: int) -> list:
    """Ubah string domain menjadi array indeks karakter — identik dengan Android tokenize()"""
    tokens = [PAD_IDX] * max_len
    for i, char in enumerate(domain[:max_len]):
        tokens[i] = CHAR_TO_IDX.get(char, UNK_IDX)
    return tokens

# Verifikasi
test_urls = [
    'https://www.youporn.com/watch/123',
    'google.com',
    'xvideos.com',
]
print(f'{"URL":<45} {"Domain":<20} {"Core"}')
print('-' * 75)
for url in test_urls:
    d  = normalize_domain(url)
    core = extract_core_domain(d)
    print(f'{url:<45} {d:<20} {core}')
    print(f'  Token[0:10]: {tokenize(core, 34)[:10]}...')

## Cell 5 — Siapkan Dataset 50K untuk Tuning

Tuning menggunakan **50K sampel** (25K per kelas) agar lebih cepat.

In [ ]:
N_TUNE_PER_CLASS = 25_000  # 25K aman + 25K porno = 50K total

df_safe = df[df[LABEL_COL] == 0].sample(n=N_TUNE_PER_CLASS, random_state=42)
df_porn = df[df[LABEL_COL] == 1].sample(n=N_TUNE_PER_CLASS, random_state=42)
df_tune = pd.concat([df_safe, df_porn]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Dataset tuning: {len(df_tune):,} baris')
print('Membangun sekuens karakter...')

# Hitung MAX_LEN dari persentil 95 panjang domain
all_domains = df_tune[URL_COL].apply(normalize_domain)
domain_lengths = all_domains.apply(len)
MAX_LEN = int(np.percentile(domain_lengths, 95))
MAX_LEN = max(MAX_LEN, 20)  # Minimal 20 karakter
MAX_LEN = min(MAX_LEN, 100) # Maksimal 100 karakter

print(f'\nAnalisis panjang domain:')
print(f'  Min    : {domain_lengths.min()}')
print(f'  Median : {domain_lengths.median():.0f}')
print(f'  P95    : {np.percentile(domain_lengths, 95):.0f}')
print(f'  Max    : {domain_lengths.max()}')
print(f'  MAX_LEN yang dipakai: {MAX_LEN}')

# Bangun sequences
X_list, y_list = [], []

for _, row in df_tune.iterrows():
    label       = int(row[LABEL_COL])
    full_domain = normalize_domain(str(row[URL_COL]))

    if not full_domain or '.' not in full_domain:
        continue

    X_list.append(tokenize(full_domain, MAX_LEN))
    y_list.append(label)

X = np.array(X_list, dtype=np.int32)
y = np.array(y_list, dtype=np.float32)

print(f'\nTotal sampel: {len(X):,}')
print(f'Shape X: {X.shape}  (sampel × karakter)')
print(f'Distribusi: aman={int((y==0).sum()):,}, porno={int((y==1).sum()):,}')

# Split 80/20
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'\nTrain: {len(X_train):,} | Val: {len(X_val):,}')

# Distribusi kelas per split
print(f'\nDistribusi Kelas per Split:')
for split_name, y_split in [('Train', y_train), ('Val', y_val)]:
    n_aman  = int((y_split == 0).sum())
    n_porno = int((y_split == 1).sum())
    total   = len(y_split)
    print(f'  {split_name:<8}: aman={n_aman:,}, porno={n_porno:,} ({n_porno/total*100:.1f}%)')

# Hitung class weights untuk training
classes = np.unique(y_train.astype(int))
weights = compute_class_weight('balanced', classes=classes, y=y_train.astype(int))
class_weights_dict = {int(c): float(w) for c, w in zip(classes, weights)}
print(f'\nClass Weights (balanced):')
for cls, w in class_weights_dict.items():
    lbl = 'Aman (0)' if cls == 0 else 'Porno (1)'
    print(f'  {lbl}: {w:.4f}')

## Cell 6 — Definisi Model CNN-1D (Grid Search)

Fungsi `build_cnn1d(num_filters, kernel_size, dropout_rate, learning_rate)` mendefinisikan
arsitektur CNN-1D Sequential yang menerima hyperparameter secara eksplisit — tanpa Keras Tuner.
Ini memungkinkan kita menyimpan **plot training per eksperimen** secara langsung.

**Arsitektur tetap (tidak dituning):**
- `embed_dim = 32`: dimensi embedding karakter
- `dense_units = 64`: ukuran Fully Connected Layer

**Hyperparameter yang dituning (Tabel III.3):**
- `num_filters`: [32, 64, 128] — kapasitas ekstraksi fitur Conv1D
- `kernel_size`: [3, 5, 7] — cakupan pola lokal (trigram, pentagram, heptagram)
- `dropout_rate`: [0.1, 0.2, 0.5] — tingkat regularisasi
- `learning_rate`: [0.001, 0.0001] — laju konvergensi optimizer Adam

In [ ]:
def build_cnn1d(num_filters: int, kernel_size: int, dropout_rate: float, learning_rate: float) -> tf.keras.Model:
    """
    Builder CNN-1D Sequential (satu lapisan konvolusi).
    Menerima hyperparameter langsung — tidak bergantung pada Keras Tuner.
    """
    EMBED_DIM   = 32
    DENSE_UNITS = 64

    inputs = tf.keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name='input')

    x = tf.keras.layers.Embedding(
        input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
        input_length=MAX_LEN, name='embedding'
    )(inputs)

    x = tf.keras.layers.Conv1D(
        filters=num_filters, kernel_size=kernel_size,
        activation='relu', padding='same', name='conv1d'
    )(x)

    x = tf.keras.layers.MaxPooling1D(name='maxpool')(x)

    x = tf.keras.layers.Dropout(dropout_rate, name='dropout')(x)

    x = tf.keras.layers.GlobalMaxPooling1D(name='global_maxpool')(x)

    x = tf.keras.layers.Dense(DENSE_UNITS, activation='relu', name='dense')(x)

    output = tf.keras.layers.Dense(1, activation='sigmoid', name='output')(x)

    model = tf.keras.Model(inputs=inputs, outputs=output)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

# Preview model dengan parameter contoh
preview = build_cnn1d(num_filters=64, kernel_size=5, dropout_rate=0.2, learning_rate=1e-3)
preview.summary()
print(f'\nTotal parameter: {preview.count_params():,}')
del preview
tf.keras.backend.clear_session()

## Cell 7 — Jalankan Grid Search

**Grid Search** mencoba **semua 54 kombinasi** hyperparameter secara exhaustive:
1. Menghasilkan semua 54 kombinasi dengan `itertools.product`
2. Melatih setiap kombinasi dan menyimpan **plot + CSV per eksperimen**
3. Memilih kombinasi dengan **validation accuracy tertinggi** sebagai yang terbaik

Kelebihan dibanding Keras Tuner:
- Bisa menyimpan `training_history.png`, `metrics_barchart.png`, `epoch_history.csv`, dan `experiment_info.json` **per eksperimen**
- Kontrol penuh atas reproducibility dan logging

**Konfigurasi:**
- Total kombinasi    : 54 (3 x 3 x 3 x 2) — semua dicoba
- EPOCHS_PER_TRIAL  : maks 100 (early stopping memantau val_loss, patience=3)
- Metrik seleksi    : val_accuracy (sesuai laporan Subbab III.6.4)

> Estimasi waktu: **45–90 menit** (tergantung GPU Colab).

In [ ]:
# ── Konfigurasi Grid Search ───────────────────────────────────────
PARAM_SPACE = {
    'num_filters'  : [32, 64, 128],
    'kernel_size'  : [3, 5, 7],
    'dropout_rate' : [0.1, 0.2, 0.5],
    'learning_rate': [1e-3, 1e-4],
}
EPOCHS_PER_TRIAL = 100
BATCH_SIZE_TUNE  = 256

TIMESTAMP        = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
TUNING_PLOTS_DIR = SAVE_PATH + f'CNN1D_Tuning_Plots_{TIMESTAMP}/'
os.makedirs(TUNING_PLOTS_DIR, exist_ok=True)
print(f'Output folder: {TUNING_PLOTS_DIR}')

# Grid Search — coba semua 54 kombinasi
all_combos      = list(itertools.product(*PARAM_SPACE.values()))
total_kombinasi = len(all_combos)
selected        = all_combos

print(f'Total kombinasi : {total_kombinasi}  (3x3x3x2 = 54)')
print(f'Yang dicoba     : semua {len(selected)} kombinasi (Grid Search)')
print(f'Epochs per trial: maks {EPOCHS_PER_TRIAL} (early stopping patience=3)')
print('-' * 60)

tuning_results = []
best_val_acc   = 0.0
best_params    = None
tuning_start   = time.time()

for idx, (num_filters, kernel_size, dropout_rate, learning_rate) in enumerate(selected):
    name = f'nf{num_filters}_ks{kernel_size}_dr{dropout_rate}_lr{learning_rate}'
    print(f'\n[{idx+1:02d}/{len(selected)}] {name}')

    # ── Bangun dan latih model ────────────────────────────────────
    tf.keras.backend.clear_session()
    model = build_cnn1d(num_filters, kernel_size, dropout_rate, learning_rate)

    early_stop = tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=3, restore_best_weights=True, verbose=0
    )

    t0   = time.time()
    hist = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS_PER_TRIAL,
        batch_size=BATCH_SIZE_TUNE,
        callbacks=[early_stop],
        class_weight=class_weights_dict,
        verbose=1
    )
    train_time = time.time() - t0

    # ── Hitung metrik pada val set ────────────────────────────────
    y_pred_prob = model.predict(X_val, verbose=0).flatten()
    y_pred_bin  = (y_pred_prob >= 0.5).astype(int)
    y_val_int   = y_val.astype(int)

    val_acc  = float(accuracy_score(y_val_int, y_pred_bin))
    val_f1   = float(f1_score(y_val_int, y_pred_bin, zero_division=0))
    val_prec = float(precision_score(y_val_int, y_pred_bin, zero_division=0))
    val_rec  = float(recall_score(y_val_int, y_pred_bin, zero_division=0))
    n_epochs = len(hist.history['loss'])

    is_best = val_acc > best_val_acc
    if is_best:
        best_val_acc = val_acc
        best_params  = {
            'name'         : name,
            'num_filters'  : num_filters,
            'kernel_size'  : kernel_size,
            'dropout_rate' : dropout_rate,
            'learning_rate': learning_rate,
            'embed_dim'    : 32,
            'dense_units'  : 64,
            'max_len'      : MAX_LEN,
            'vocab_size'   : VOCAB_SIZE,
        }
        model.save_weights('/content/best_cnn1d_weights.weights.h5')
        print(f'  ★ NEW BEST  val_acc={val_acc:.4f}')

    r = {
        'name'         : name,
        'num_filters'  : num_filters,
        'kernel_size'  : kernel_size,
        'dropout_rate' : dropout_rate,
        'learning_rate': learning_rate,
        'val_accuracy' : round(val_acc,  4),
        'val_f1'       : round(val_f1,   4),
        'val_precision': round(val_prec, 4),
        'val_recall'   : round(val_rec,  4),
        'n_epochs'     : n_epochs,
        'train_time_s' : round(train_time, 1),
        'is_best'      : is_best,
    }
    tuning_results.append(r)

    # ── Simpan per-experiment plots & files ───────────────────────
    exp_folder = TUNING_PLOTS_DIR + f'exp{idx+1:02d}_{name}/'
    os.makedirs(exp_folder, exist_ok=True)

    # 1. training_history.png
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(f'Eksperimen {idx+1:02d}: {name}', fontsize=10)
    axes[0].plot(hist.history['loss'],     label='Train Loss')
    axes[0].plot(hist.history['val_loss'], label='Val Loss')
    axes[0].set_title('Loss per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(hist.history['accuracy'],     label='Train Acc')
    axes[1].plot(hist.history['val_accuracy'], label='Val Acc')
    axes[1].set_title('Accuracy per Epoch'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(exp_folder + 'training_history.png', dpi=120, bbox_inches='tight')
    plt.close()

    # 2. metrics_barchart.png
    fig, ax = plt.subplots(figsize=(6, 4))
    m_names  = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
    m_values = [val_acc, val_f1, val_prec, val_rec]
    bar_c    = ['#1E88E5', '#43A047', '#FB8C00', '#E53935']
    bars = ax.bar(m_names, m_values, color=bar_c, alpha=0.85)
    for bar, val in zip(bars, m_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, 1.1); ax.set_title(f'Metrik Validasi — Exp {idx+1:02d}')
    ax.set_ylabel('Skor'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(exp_folder + 'metrics_barchart.png', dpi=120, bbox_inches='tight')
    plt.close()

    # 3. epoch_history.csv
    hist_df = pd.DataFrame(hist.history)
    hist_df.index.name = 'epoch'
    hist_df.to_csv(exp_folder + 'epoch_history.csv')

    # 4. experiment_info.json
    exp_info = {
        'experiment_id'  : idx + 1,
        'name'           : name,
        'hyperparameters': {
            'num_filters'  : num_filters,
            'kernel_size'  : kernel_size,
            'dropout_rate' : dropout_rate,
            'learning_rate': learning_rate,
            'embed_dim'    : 32,
            'dense_units'  : 64,
        },
        'metrics': {
            'val_accuracy' : round(val_acc,  4),
            'val_f1'       : round(val_f1,   4),
            'val_precision': round(val_prec, 4),
            'val_recall'   : round(val_rec,  4),
        },
        'training_info': {
            'n_epochs_run' : n_epochs,
            'max_epochs'   : EPOCHS_PER_TRIAL,
            'train_time_s' : round(train_time, 1),
            'batch_size'   : BATCH_SIZE_TUNE,
            'early_stopped': n_epochs < EPOCHS_PER_TRIAL,
        },
        'is_best': is_best,
    }
    with open(exp_folder + 'experiment_info.json', 'w') as f:
        json.dump(exp_info, f, indent=2)

    del model
    print(f'  acc={val_acc:.4f}  f1={val_f1:.4f}  prec={val_prec:.4f}  rec={val_rec:.4f}  '
          f'epoch={n_epochs}  {train_time:.0f}s')

total_tuning_time = time.time() - tuning_start
print(f'\n{"="*60}')
print(f'Grid Search selesai dalam {total_tuning_time/60:.1f} menit')
print(f'Best val_accuracy: {best_val_acc:.4f}')
print(f'Best params      : {best_params["name"]}')

## Cell 8 — Hasil Tuning

In [ ]:
# Tampilkan semua hasil tuning diurutkan val_accuracy tertinggi di atas
print('Hasil semua skenario tuning (diurutkan val_accuracy ↓):')
print('=' * 95)
header = f'{"Rank":<5} {"Konfigurasi":<45} {"Acc":>6} {"F1":>6} {"Prec":>6} {"Rec":>6} {"Ep":>3} {"Waktu":>6}'
print(header)
print('-' * 95)

for rank, r in enumerate(sorted(tuning_results, key=lambda x: x['val_accuracy'], reverse=True), 1):
    marker = ' ★' if r['is_best'] else ''
    print(f'{rank:<5} {r["name"]:<45} '
          f'{r["val_accuracy"]:>6.4f} {r["val_f1"]:>6.4f} '
          f'{r["val_precision"]:>6.4f} {r["val_recall"]:>6.4f} '
          f'{r["n_epochs"]:>3} {r["train_time_s"]:>5.0f}s'
          f'{marker}')

print('=' * 95)
print(f'\nHyperparameter terbaik (val_accuracy={best_val_acc:.4f}):')
print(json.dumps({k: v for k, v in best_params.items() if k not in ('name', 'is_best')}, indent=2))

## Cell 9 — Evaluasi Model Terbaik pada Validation Set

In [ ]:
# Bangun ulang model terbaik dan load bobot yang sudah disimpan
print(f'Membangun ulang model terbaik: {best_params["name"]}')
best_model = build_cnn1d(
    num_filters   = best_params['num_filters'],
    kernel_size   = best_params['kernel_size'],
    dropout_rate  = best_params['dropout_rate'],
    learning_rate = best_params['learning_rate'],
)
best_model.load_weights('/content/best_cnn1d_weights.weights.h5')

# Evaluasi pada validation set
y_pred_prob = best_model.predict(X_val, verbose=0).flatten()
y_pred      = (y_pred_prob >= 0.5).astype(int)
y_val_int   = y_val.astype(int)

val_acc  = accuracy_score(y_val_int, y_pred)
val_f1   = f1_score(y_val_int, y_pred, zero_division=0)
val_prec = precision_score(y_val_int, y_pred, zero_division=0)
val_rec  = recall_score(y_val_int, y_pred, zero_division=0)

print(f'\nEvaluasi Model Terbaik pada Validation Set:')
print('=' * 50)
print(f'  Accuracy  : {val_acc:.4f}')
print(f'  F1-Score  : {val_f1:.4f}')
print(f'  Precision : {val_prec:.4f}')
print(f'  Recall    : {val_rec:.4f}')
print('=' * 50)
print(classification_report(y_val_int, y_pred,
      target_names=['Aman (0)', 'Pornografi (1)'], digits=4))

# Plot distribusi skor prediksi
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(y_pred_prob[y_val == 0], bins=50, alpha=0.6, label='URL Aman', color='green')
ax.hist(y_pred_prob[y_val == 1], bins=50, alpha=0.6, label='URL Porno', color='red')
ax.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
ax.set_xlabel('Skor Prediksi (P_porno)')
ax.set_ylabel('Frekuensi')
ax.set_title(f'Distribusi Skor — Best Model: {best_params["name"]}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(TUNING_PLOTS_DIR + 'best_model_score_dist.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 10 — Simpan Hyperparameter Terbaik ke JSON
File ini akan dibaca otomatis oleh notebook training.

In [ ]:
# Simpan best_params ke JSON (tanpa field 'name' dan 'is_best')
best_hp_params = {k: v for k, v in best_params.items() if k not in ('name', 'is_best')}
json_path = SAVE_PATH + 'best_params_cnn.json'
with open(json_path, 'w') as f:
    json.dump(best_hp_params, f, indent=2)

print(f'✅ best_params_cnn.json tersimpan: {json_path}')
print('\nIsi file:')
print(json.dumps(best_hp_params, indent=2))
print('\n--- SELESAI ---')
print('Langkah berikutnya: Buka notebook cnn1d_training.ipynb')

## Cell 11 — Ekspor CSV, Visualisasi Perbandingan, dan Resume Tuning
Simpan tabel hasil semua skenario ke CSV, buat horizontal bar chart (merah = terbaik),
dan generate `resume_tuning_cnn1d.md` yang merangkum seluruh proses tuning.

In [ ]:
# ── 1. Simpan CSV dari tuning_results ────────────────────────────
tuning_df        = pd.DataFrame(tuning_results)
tuning_df_sorted = tuning_df.sort_values('val_accuracy', ascending=False).reset_index(drop=True)
tuning_df_sorted.insert(0, 'rank', range(1, len(tuning_df_sorted) + 1))

csv_path = TUNING_PLOTS_DIR + 'cnn1d_tuning_results.csv'
tuning_df_sorted.to_csv(csv_path, index=False)
print(f'✅ CSV tersimpan: {csv_path}')
print(f'   Shape: {tuning_df_sorted.shape}')
print()
display(tuning_df_sorted[['rank','name','val_accuracy','val_f1','val_precision','val_recall','n_epochs','train_time_s']])

# ── 2. Horizontal Bar Chart Perbandingan ─────────────────────────
sorted_asc = tuning_df.sort_values('val_accuracy', ascending=True)
best_name  = best_params['name']
bar_colors = ['#D32F2F' if r == best_name else '#1E88E5' for r in sorted_asc['name']]

fig, ax = plt.subplots(figsize=(10, max(6, len(sorted_asc) * 0.4)))
bars = ax.barh(sorted_asc['name'], sorted_asc['val_accuracy'],
               color=bar_colors, alpha=0.85, height=0.7)
for bar, val in zip(bars, sorted_asc['val_accuracy']):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', ha='left', fontsize=8)
ax.set_xlabel('Val Accuracy')
ax.set_title('CNN-1D Tuning — Perbandingan Semua Skenario\n(merah = konfigurasi terbaik)')
ax.set_xlim(0, min(1.08, sorted_asc['val_accuracy'].max() + 0.08))
ax.axvline(best_val_acc, color='#D32F2F', linestyle='--', alpha=0.5,
           label=f'Best: {best_val_acc:.4f}')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
chart_path = TUNING_PLOTS_DIR + 'cnn1d_tuning_comparison.png'
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Comparison chart tersimpan: {chart_path}')

# ── 3. Generate resume_tuning_cnn1d.md ───────────────────────────
total_min  = total_tuning_time / 60
best_hp    = {k: v for k, v in best_params.items() if k not in ('name', 'is_best')}

ranking_rows = ''
for rank, r in enumerate(sorted(tuning_results, key=lambda x: x['val_accuracy'], reverse=True), 1):
    marker = ' ★' if r['is_best'] else ''
    ranking_rows += (
        f'| {rank} | `{r["name"]}` | {r["val_accuracy"]:.4f} | {r["val_f1"]:.4f} | '
        f'{r["val_precision"]:.4f} | {r["val_recall"]:.4f} | {r["n_epochs"]} | '
        f'{r["train_time_s"]:.0f}s |{marker}\n'
    )

best_hp_rows = ''
for k, v in best_hp.items():
    best_hp_rows += f'| `{k}` | {v} |\n'

resume_lines = []
resume_lines.append('# Resume Hyperparameter Tuning — CNN-1D URL Classifier\n')
resume_lines.append('\n## Konfigurasi Tuning\n\n')
resume_lines.append('| Parameter | Nilai |\n')
resume_lines.append('|-----------|-------|\n')
resume_lines.append('| Metode | Grid Search |\n')
resume_lines.append(f'| Total kombinasi | {total_kombinasi} (3x3x3x2 = 54, semua dicoba) |\n')
resume_lines.append(f'| EPOCHS_PER_TRIAL | {EPOCHS_PER_TRIAL} (EarlyStopping patience=3) |\n')
resume_lines.append('| Seed | 42 (PYTHONHASHSEED=0, random, numpy, tensorflow) |\n')
resume_lines.append(f'| Total waktu tuning | {total_min:.1f} menit |\n')
resume_lines.append(f'| Timestamp | {TIMESTAMP} |\n')
resume_lines.append('\n## Ruang Pencarian Hyperparameter\n\n')
resume_lines.append('| Hyperparameter | Nilai yang Dicoba |\n')
resume_lines.append('|----------------|------------------|\n')
resume_lines.append('| `num_filters` | 32, 64, 128 |\n')
resume_lines.append('| `kernel_size` | 3, 5, 7 |\n')
resume_lines.append('| `dropout_rate` | 0.1, 0.2, 0.5 |\n')
resume_lines.append('| `learning_rate` | 0.001, 0.0001 |\n')
resume_lines.append('\n## Hyperparameter Terbaik\n\n')
resume_lines.append('| Parameter | Nilai |\n')
resume_lines.append('|-----------|-------|\n')
resume_lines.append(best_hp_rows)
resume_lines.append(f'\n**Best val_accuracy: {best_val_acc:.4f}**\n')
resume_lines.append('\n## Ranking Semua Skenario\n\n')
resume_lines.append('| Rank | Konfigurasi | Accuracy | F1 | Precision | Recall | Epochs | Waktu |\n')
resume_lines.append('|------|-------------|----------|----|-----------|--------|--------|-------|\n')
resume_lines.append(ranking_rows)
resume_lines.append('\n★ = konfigurasi terbaik\n')
resume_lines.append('\n## Output Files\n\n')
resume_lines.append('- `cnn1d_tuning_results.csv` — tabel semua skenario\n')
resume_lines.append('- `cnn1d_tuning_comparison.png` — horizontal bar chart perbandingan\n')
resume_lines.append('- `best_model_score_dist.png` — distribusi skor model terbaik\n')
resume_lines.append('- `exp{NN}_{name}/training_history.png` — training/val loss & accuracy per eksperimen\n')
resume_lines.append('- `exp{NN}_{name}/metrics_barchart.png` — bar chart metrik per eksperimen\n')
resume_lines.append('- `exp{NN}_{name}/epoch_history.csv` — log per epoch\n')
resume_lines.append('- `exp{NN}_{name}/experiment_info.json` — metadata eksperimen\n')

resume_md   = ''.join(resume_lines)
resume_path = TUNING_PLOTS_DIR + 'resume_tuning_cnn1d.md'
with open(resume_path, 'w', encoding='utf-8') as f:
    f.write(resume_md)

print(f'\n✅ Resume tersimpan: {resume_path}')
print('\n--- TUNING SELESAI ---')
print(f'Best config   : {best_params["name"]}')
print(f'Best val_acc  : {best_val_acc:.4f}')
print(f'Output folder : {TUNING_PLOTS_DIR}')